# BTP: Optical Transient Classification — Balanced Hierarchical Pipeline

**Classes (8):** SN Ia, SN Ib, SN Ic, SN II, SLSN, AGN, TDE (TNS, spectroscopically
confirmed, photometry via ZTF/ALeRCE) and Stellar Flares (TESS, via lightkurve).

**Two-stage hierarchy**
- **Stage 1 (coarse):** SNe vs AGN vs TDE vs Stellar Flare — 150 objects each, 600 total.
- **Stage 2 (fine):** for SNe, which subtype — Ia / Ib / Ic / II / SLSN, 30 each.

**Four models at both stages:** Random Forest, Logistic Regression, Bagging (Trees),
Bagging (SVM).

---

### What changed in this rebuild, and why

1. **Balanced acquisition by retry, not by hope.** The previous pipeline sampled a
   fixed number of candidates per class and accepted whatever survived cross-match,
   download and GP fitting. Survival is not random — sparser, noisier classes fail
   more often — so the SN subtype set collapsed to 95 objects (Ia 18, Ib 13, Ic 15,
   II 21, SLSN 28) with the rare classes hit hardest, which is the direct cause of
   Stage 2 landing near 47-53%. Every class now uses a *keep pulling candidates
   until N survive* loop, so the final counts are exact.

2. **One global split, not two independent ones.** Earlier versions split Stage 1
   (over the full dataset) and Stage 2 (over the SN subset) independently, which let
   an SN object in Stage 1's test set turn up in Stage 2's training set and quietly
   inflate the cascaded accuracy. There is now exactly **one** stratified split;
   Stage 2's train and test rows are subsets of Stage 1's own partitions **by
   construction**, and that nesting is asserted in code. No "leakage-free retrain"
   cleanup step is needed, because leakage is structurally impossible.

3. **A fourth model, configured for what it is.** Bagging (SVM) does not inherit the
   tree-bagging settings — see the model-building cell for the reasoning.

4. **One importance method that spans all four families.** Permutation importance,
   computed identically for every model on held-out data, because an RBF-kernel
   bagged SVM has neither coefficients nor impurity gains.

Code that the rebuild did not need to change — `classify_sn_subtype`,
`extract_ztf_name`, `resolve_oid`, `clean_lightcurve`, `gp_interpolate`,
`extract_shape_features`, `process_ztf_object`, `process_tess_object` — is reused
unchanged from the previous notebook.

# PHASE 2 — Data Acquisition (TNS + ZTF/ALeRCE + TESS)

In [ ]:
# Setup — install, mount, config
!pip install -q alerce lightkurve astroquery pandas numpy matplotlib tqdm requests

import os, io, json, time, zipfile, requests, gc
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from alerce.core import Alerce

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/BTP_optical_transients/data/raw'
FEATURE_DIR = '/content/drive/MyDrive/BTP_optical_transients/data/features'
for sub in ['sn_ia', 'sn_ib', 'sn_ic', 'sn_ii', 'slsn', 'agn', 'tde', 'stellar_flares', 'tns_catalog']:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)
os.makedirs(FEATURE_DIR, exist_ok=True)

alerce = Alerce()

# AGN / TDE / Stellar Flares target this many objects each
N_PER_CLASS = 150
# Each SN subtype (Ia/Ib/Ic/II/SLSN) targets this many objects — kept smaller
# since fine-grained subtyping has far less available data per subtype on TNS
N_PER_SN_SUBTYPE = 30

print('Data root:', BASE_DIR)

In [ ]:
# TNS credentials

TNS_MARKER = 'tns_marker{"tns_id":197986,"type": "bot", "name":"Arnav Deshpande "}'
TNS_API_KEY = "17862866246a7892202547c2.12318200"

HEADERS = {'user-agent': TNS_MARKER}

In [ ]:
# Download the staged TNS public objects CSV (bulk file, exactly as TNS's approval
# email and docs ask for — NOT per-object queries, to keep load on their servers low)
TNS_CSV_URL = "https://www.wis-tns.org/system/files/tns_public_objects/tns_public_objects.csv.zip"

resp = requests.post(TNS_CSV_URL, headers=HEADERS, data={'api_key': TNS_API_KEY}, timeout=120)
resp.raise_for_status()

zip_path = os.path.join(BASE_DIR, 'tns_catalog', 'tns_public_objects.csv.zip')
with open(zip_path, 'wb') as f:
    f.write(resp.content)
with zipfile.ZipFile(zip_path) as z:
    csv_name = z.namelist()[0]
    z.extractall(os.path.join(BASE_DIR, 'tns_catalog'))

# First line of the file is a timestamp, not the header — confirmed against TNS's own docs
tns_full = pd.read_csv(os.path.join(BASE_DIR, 'tns_catalog', csv_name), skiprows=1)
print(tns_full.shape)
print(tns_full.columns.tolist())
tns_full.head()

In [ ]:
# Filter to SN Ia/Ib/Ic/II/SLSN, AGN and TDE, check class balance BEFORE downloading
print(tns_full['type'].value_counts().head(30))

def classify_sn_subtype(t):
    """
    Maps a raw TNS `type` string to one of five SN subtype buckets, or None if it
    doesn't belong to any of them. SLSN-I/II/R are TNS's own top-level type strings
    (not prefixed 'SN '), so they're checked separately from the SN Ia/Ib/Ic/II family.
    Sub-subtypes are folded into their parent bucket for a manageable class count:
    e.g. 'SN Ia-91bg', 'SN Ia-pec', 'SN Iax' -> SN_Ia; 'SN Ic-BL', 'SN Icn' -> SN_Ic;
    'SN IIb', 'SN IIn', 'SN IIP', 'SN IIL' -> SN_II.
    """
    t = str(t)
    if 'SLSN' in t:
        return 'SLSN'
    if not t.startswith('SN '):
        return None
    rest = t[3:]
    if rest.startswith('Ia'):
        return 'SN_Ia'
    if rest.startswith('Ib'):
        return 'SN_Ib'
    if rest.startswith('Ic'):
        return 'SN_Ic'
    if rest.startswith('II'):
        return 'SN_II'
    return None

tns_full['sn_subtype'] = tns_full['type'].apply(classify_sn_subtype)
agn_mask = tns_full['type'].astype(str).str.contains('AGN', case=False, na=False)
tde_mask = tns_full['type'].astype(str).str.contains('TDE', case=False, na=False)

SN_SUBTYPES = ['SN_Ia', 'SN_Ib', 'SN_Ic', 'SN_II', 'SLSN']
KEEP_COLS = ['name', 'ra', 'declination', 'redshift', 'type', 'discoverydate', 'internal_names']

tns_subtype_dfs = {}
for subtype in SN_SUBTYPES:
    df = tns_full[tns_full['sn_subtype'] == subtype].copy()
    df = df[[c for c in KEEP_COLS if c in df.columns]]
    print(f"TNS {subtype} rows available: {len(df)}")
    if len(df) < N_PER_SN_SUBTYPE:
        print(f"  NOTE: only {len(df)} available, below the target of {N_PER_SN_SUBTYPE} — "
              f"this class will end up smaller than the others.")
    df = df.sample(min(N_PER_SN_SUBTYPE, len(df)), random_state=42)
    df.to_csv(os.path.join(BASE_DIR, 'tns_catalog', f'tns_{subtype.lower()}_labels.csv'), index=False)
    tns_subtype_dfs[subtype] = df

tns_sn_ia = tns_subtype_dfs['SN_Ia']
tns_sn_ib = tns_subtype_dfs['SN_Ib']
tns_sn_ic = tns_subtype_dfs['SN_Ic']
tns_sn_ii = tns_subtype_dfs['SN_II']
tns_slsn  = tns_subtype_dfs['SLSN']

tns_agn = tns_full[agn_mask][[c for c in KEEP_COLS if c in tns_full.columns]].copy()
tns_tde = tns_full[tde_mask][[c for c in KEEP_COLS if c in tns_full.columns]].copy()
print(f"TNS AGN rows: {len(tns_agn)}")
print(f"TNS TDE rows: {len(tns_tde)}")
if len(tns_agn) < N_PER_CLASS:
    print(f"NOTE: only {len(tns_agn)} AGN available, below the target of {N_PER_CLASS}.")
if len(tns_tde) < N_PER_CLASS:
    print(f"NOTE: only {len(tns_tde)} TDE available, below the target of {N_PER_CLASS}.")

tns_agn = tns_agn.sample(min(N_PER_CLASS, len(tns_agn)), random_state=42)
tns_tde = tns_tde.sample(min(N_PER_CLASS, len(tns_tde)), random_state=42)
tns_agn.to_csv(os.path.join(BASE_DIR, 'tns_catalog', 'tns_agn_labels.csv'), index=False)
tns_tde.to_csv(os.path.join(BASE_DIR, 'tns_catalog', 'tns_tde_labels.csv'), index=False)

tns_sn_ia.head()

In [ ]:
def extract_ztf_name(internal_names):
    if not isinstance(internal_names, str):
        return None
    for tok in internal_names.split(','):
        tok = tok.strip()
        if tok.startswith('ZTF'):
            return tok
    return None

def resolve_oid(row, radius_arcsec=2.0):
    ztf_name = extract_ztf_name(row.get('internal_names'))
    if ztf_name:
        return ztf_name
    try:
        res = alerce.query_objects(ra=row['ra'], dec=row['declination'],
                                    radius=radius_arcsec, format='pandas')
        if len(res):
            return res.iloc[0]['oid']
    except Exception:
        pass
    return None

N_PER_SUBTYPE_TARGET = 30
OVERSAMPLE_FACTOR = 3
N_PER_SUBTYPE_SAMPLE = N_PER_SUBTYPE_TARGET * OVERSAMPLE_FACTOR

In [ ]:
FLARE_STAR_NAMES = [
    # --- Standard highly active UV Ceti types ---
    "UV Ceti", "AD Leo", "EV Lac", "YZ CMi", "AU Mic", "Proxima Centauri",
    "GJ 1243", "Ross 154", "Ross 128", "Wolf 359", "Lalande 21185", "Kruger 60",
    "TZ Ari", "GJ 1111", "V1216 Sgr", "EQ Peg", "DO Cep", "V1005 Ori",

    # --- Additional active M-dwarfs and Kepler/TESS targets ---
    "V371 Ori", "WX UMa", "Luyten 726-8", "GJ 896A", "GJ 1156", "GJ 1245A",
    "GJ 3236", "GJ 3338", "GJ 3737", "GJ 3685A", "GJ 424", "GJ 1002",
    "TRAPPIST-1", "LHS 1140", "Teegarden's Star", "GJ 1061", "YZ Cet",
    "Luyten's Star", "Lacaille 8760", "Lacaille 9352", "Gliese 1", "Gliese 876",
    "Gliese 682", "Gliese 832", "Gliese 667 C", "Kepler-411",
]

## Preprocessing and feature extraction

These are unchanged from the previous notebook. They are defined *before*
acquisition now, because the retry loop calls the feature extractor inline: an
object only counts towards the target once it has actually produced a feature row.
That is the whole mechanism by which the final counts come out exact.

GP interpolation downsamples to at most 800 points before fitting — real TESS light
curves run ~13,000-20,000 points/sector, which was the root cause of an earlier OOM
crash (verified: 18k points killed the kernel instantly, 800 points fits in ~2s).

In [ ]:
# Cleaning helper
def clean_lightcurve(df, value_col, err_col=None, min_points=8, sigma_clip=5.0):
    df = df.dropna(subset=[value_col]).copy()
    if len(df) < min_points:
        return None
    med = df[value_col].median()
    mad = (df[value_col] - med).abs().median() * 1.4826 + 1e-6
    df = df[(df[value_col] - med).abs() < sigma_clip * mad]
    if len(df) < min_points:
        return None
    return df.sort_values(df.columns[0]).reset_index(drop=True)

In [ ]:
# GP interpolation — downsamples before fitting (root-caused fix, see notes above).
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C

MAX_GP_POINTS = 800   # tested: 18k points -> OOM killed instantly; 800 points -> fits in ~2s

def downsample_for_gp(t, y, max_points=MAX_GP_POINTS):
    if len(t) <= max_points:
        return t, y
    idx = np.linspace(0, len(t) - 1, max_points).astype(int)
    return t[idx], y[idx]

def gp_interpolate(t, y, n_points=200):
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)
    t, y = downsample_for_gp(t, y)
    t0 = t.min()
    X = (t - t0).reshape(-1, 1)
    span = max(t.max() - t.min(), 1.0)
    kernel = (C(1.0, (1e-3, 1e3))
              * RBF(length_scale=span / 10, length_scale_bounds=(1e-2, 1e3))
              + WhiteKernel(noise_level=0.05, noise_level_bounds=(1e-5, 2.0)))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=1)
    gp.fit(X, y)
    t_grid_rel = np.linspace(0, t.max() - t0, n_points)
    y_grid, y_std = gp.predict(t_grid_rel.reshape(-1, 1), return_std=True)
    return t_grid_rel + t0, y_grid

In [ ]:
# Shape-feature extractor
def extract_shape_features(t_grid, y_grid):
    peak_val = float(y_grid.max())
    min_val  = float(y_grid.min())
    t_peak   = float(t_grid[np.argmax(y_grid)])
    rise_time  = t_peak - t_grid[0]
    decay_time = t_grid[-1] - t_peak
    amplitude  = peak_val - min_val
    return {'peak_val': peak_val, 'rise_time': rise_time, 'decay_time': decay_time, 'amplitude': amplitude}

In [ ]:
# SN subtypes / AGN / TDE feature extraction (checkpointed, resumable)
def mag_to_relflux(mag, baseline_mag):
    return 10 ** (-0.4 * (mag - baseline_mag))

def process_ztf_object(path, oid, label):
    df = pd.read_csv(path)
    row = {'id': oid, 'label': label, 'survey': 'ZTF'}
    band_curves = {}
    for fid, band_name in [(1, 'g'), (2, 'r')]:
        band_df = clean_lightcurve(df[df['fid'] == fid][['mjd', 'magpsf']], 'magpsf', min_points=6)
        if band_df is None:
            continue
        try:
            t_grid, mag_grid = gp_interpolate(band_df['mjd'].values, band_df['magpsf'].values)
            band_curves[band_name] = (t_grid, mag_grid)
        except Exception:
            continue
    if 'r' in band_curves:
        primary_t, primary_mag = band_curves['r']
    elif 'g' in band_curves:
        primary_t, primary_mag = band_curves['g']
    else:
        return None
    baseline_mag = primary_mag.max()
    relflux = mag_to_relflux(primary_mag, baseline_mag)
    row.update(extract_shape_features(primary_t, relflux))
    if 'g' in band_curves and 'r' in band_curves:
        t_peak_idx = np.argmax(relflux)
        t_peak = primary_t[t_peak_idx]
        g_t, g_mag = band_curves['g']
        r_t, r_mag = band_curves['r']
        row['color_g_r'] = float(np.interp(t_peak, g_t, g_mag) - np.interp(t_peak, r_t, r_mag))
    else:
        row['color_g_r'] = np.nan
    return row

In [ ]:
# Stellar flare feature extraction (checkpointed, resumable)
def process_tess_object(path, star_name, label):
    df = pd.read_csv(path)
    clean_df = clean_lightcurve(df[['bjd', 'flux']], 'flux', min_points=20)
    if clean_df is None:
        return None
    median_flux = clean_df['flux'].median()
    if median_flux == 0:
        return None
    try:
        t_grid, flux_grid = gp_interpolate(clean_df['bjd'].values, clean_df['flux'].values)
    except Exception:
        return None
    relflux = flux_grid / median_flux
    row = {'id': star_name, 'label': label, 'survey': 'TESS'}
    row.update(extract_shape_features(t_grid, relflux))
    row['color_g_r'] = np.nan
    return row

## Balanced acquisition — retry until the target is met

The generalisation of `build_subtype_to_target` to every class. Two loops, one
contract: *a candidate that fails at any stage is replaced, not absorbed as a loss.*

- `build_tns_class_to_target` covers all seven TNS/ALeRCE classes (the 5 SN
  subtypes plus AGN and TDE — same source, so it is close to a direct reuse).
- `build_flares_to_target` applies the same contract to how flares are actually
  sourced: TESS sectors via lightkurve, keyed by star + sector rather than by a TNS
  row. Flares are **not** forced through the TNS/ALeRCE path.

Both checkpoint to disk and resume, and both record failed candidates so a resumed
run does not re-attempt what already failed.

> **Note on superseded cells.** This replaces the previous notebook's
> `fetch_lightcurves_from_tns` and `fetch_tess_flares` driver loops and the
> `trim_to_target` helper. All three existed to cope with under-filled classes by
> accepting or trimming whatever arrived; the retry loop returns exactly the target,
> so there is nothing left to trim. The functions they *called* are reused unchanged.

In [ ]:
# Balanced acquisition — retry until target
# ---------------------------------------------------------------
# Inlined verbatim from btp_pipeline/acquisition.py, which is covered by the
# repository's synthetic-data test suite. Edit it there and rebuild this
# notebook (tools/build_notebook.py) rather than editing it here.
# ---------------------------------------------------------------

"""
Phase 2 — balanced acquisition.

Generalises the `build_subtype_to_target` pattern already proven on the 5 SN
subtypes to every class in the dataset:

  * `build_tns_class_to_target`  — TNS + ALeRCE classes (5 SN subtypes, AGN, TDE)
  * `build_flares_to_target`     — TESS/lightkurve stellar flares

The shared idea in both: a candidate that fails cross-match, download, cleaning
or GP fitting is *not* absorbed as a loss. We move on to the next candidate and
keep pulling until `target` objects have actually survived feature extraction.
That is what makes the final per-class counts exact instead of "whatever
happened to make it through", which is what previously gutted the rare classes.

Every external-API call site is injected as a callable so the control flow can
be exercised against synthetic fakes without touching TNS/ALeRCE/MAST.
"""

import gc
import os
import time

import pandas as pd


def _load_checkpoint(manifest_path, features_path, failed_path):
    """Resume from a partial pull, if one is on disk."""
    manifest, features, failed = [], [], set()
    if os.path.exists(manifest_path) and os.path.exists(features_path):
        manifest = pd.read_csv(manifest_path).to_dict('records')
        features = pd.read_csv(features_path).to_dict('records')
    if os.path.exists(failed_path):
        failed = set(pd.read_csv(failed_path)['candidate'].astype(str))
    return manifest, features, failed


def _save_checkpoint(manifest, features, failed, manifest_path, features_path, failed_path):
    pd.DataFrame(manifest).to_csv(manifest_path, index=False)
    pd.DataFrame(features).to_csv(features_path, index=False)
    pd.DataFrame({'candidate': sorted(failed)}).to_csv(failed_path, index=False)


def build_tns_class_to_target(
    label,
    pool,
    target,
    *,
    base_dir,
    feature_dir,
    resolve_oid_fn,
    query_detections_fn,
    process_fn,
    out_subdir=None,
    min_detections=5,
    sleep=0.15,
    checkpoint_every=10,
    progress=None,
    verbose=True,
):
    """
    Pull candidates from `pool` until exactly `target` of them survive all the
    way through feature extraction.

    Parameters
    ----------
    label : str
        Class label written into the manifest and feature rows (e.g. 'SN_Ia', 'AGN').
    pool : pandas.DataFrame
        Candidate TNS rows, already filtered to this class and already shuffled.
        Must contain at least 'name'; 'ra'/'declination'/'internal_names' are used
        by `resolve_oid_fn`.
    target : int
        Number of *surviving* objects required.
    resolve_oid_fn(row) -> str|None
        TNS row -> ZTF object id.
    query_detections_fn(oid) -> DataFrame|None
        ALeRCE detections for an object id.
    process_fn(csv_path, oid, label) -> dict|None
        Feature extractor; None means this object failed and we should try the next.

    Returns
    -------
    (manifest, features) : (list[dict], list[dict])
        Both truncated to exactly `target` when the pool was deep enough.
    """
    out_subdir = out_subdir or label.lower()
    out_dir = os.path.join(base_dir, out_subdir)
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(feature_dir, exist_ok=True)

    manifest_path = os.path.join(base_dir, f'{out_subdir}_manifest_topup.csv')
    features_path = os.path.join(feature_dir, f'{out_subdir}_features_topup.csv')
    failed_path = os.path.join(base_dir, f'{out_subdir}_failed_candidates.csv')

    manifest, features, failed = _load_checkpoint(manifest_path, features_path, failed_path)
    done_names = {str(m['tns_name']) for m in manifest}
    if features and verbose:
        print(f"Resuming {label}: {len(features)}/{target} already survived, "
              f"{len(failed)} known failures will be skipped")
    if len(features) >= target:
        return manifest[:target], features[:target]

    if verbose:
        print(f"{label}: candidate pool = {len(pool)} rows, need {target} survivors "
              f"({len(features)} already in hand)")

    bar = progress(total=target, initial=len(features), desc=f'{label} (survivors)') if progress else None
    attempted = 0
    try:
        for count, (_, row) in enumerate(pool.iterrows()):
            if len(features) >= target:
                break
            name = str(row['name'])
            if name in done_names or name in failed:
                continue
            attempted += 1

            oid = resolve_oid_fn(row)
            if oid is None:
                failed.add(name)
                continue
            try:
                det = query_detections_fn(oid)
            except Exception:
                failed.add(name)
                continue
            if det is None or len(det) < min_detections:
                failed.add(name)
                continue

            keep = [c for c in ['mjd', 'fid', 'magpsf', 'sigmapsf', 'ra', 'dec'] if c in det.columns]
            det = det[keep]
            csv_path = os.path.join(out_dir, f"{oid}.csv")
            det.to_csv(csv_path, index=False)

            feat = process_fn(csv_path, oid, label)
            if feat is None:
                # GP / cleaning failed -> try the NEXT candidate rather than eat the loss.
                failed.add(name)
                continue

            manifest.append({'tns_name': name, 'oid': oid, 'label': label,
                             'n_points': len(det), 'redshift': row.get('redshift')})
            features.append(feat)
            done_names.add(name)
            if bar is not None:
                bar.update(1)
            if sleep:
                time.sleep(sleep)

            if count % checkpoint_every == 0:
                _save_checkpoint(manifest, features, failed,
                                 manifest_path, features_path, failed_path)
                gc.collect()
    finally:
        if bar is not None:
            bar.close()
        _save_checkpoint(manifest, features, failed, manifest_path, features_path, failed_path)

    if verbose:
        if len(features) < target:
            print(f"  SHORTFALL: {label} reached only {len(features)}/{target} after exhausting a pool "
                  f"of {len(pool)} ({attempted} attempted this run, {len(failed)} cumulative failures). "
                  f"Genuine scarcity or attrition — raise the pool size and re-run; it resumes.")
        else:
            print(f"  {label}: reached target {target}/{target} "
                  f"({attempted} candidates attempted this run, {len(failed)} cumulative failures).")
    return manifest[:target], features[:target]


def build_flares_to_target(
    star_names,
    target,
    *,
    base_dir,
    feature_dir,
    search_fn,
    download_fn,
    process_fn,
    out_subdir='stellar_flares',
    max_sectors_per_star=14,
    checkpoint_every=5,
    progress=None,
    verbose=True,
):
    """
    Same "keep trying until N survive" contract as `build_tns_class_to_target`,
    adapted to how flares are actually sourced: TESS light curves via lightkurve,
    keyed by star name + sector rather than by a TNS row / ZTF oid.

    The important difference from the original `fetch_tess_flares` is that a
    sector only counts once `process_fn` has produced a feature row from it. The
    old loop counted *downloads*, so sectors that later failed cleaning or GP
    fitting silently shrank the class below its target.

    search_fn(star_name) -> sequence of downloadable entries (may be empty)
    download_fn(entry)   -> DataFrame with 'bjd','flux'(,'flux_err') or None
    process_fn(csv_path, star_name, label) -> dict|None
    """
    out_dir = os.path.join(base_dir, out_subdir)
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(feature_dir, exist_ok=True)

    manifest_path = os.path.join(base_dir, f'{out_subdir}_manifest_topup.csv')
    features_path = os.path.join(feature_dir, f'{out_subdir}_features_topup.csv')
    failed_path = os.path.join(base_dir, f'{out_subdir}_failed_candidates.csv')

    manifest, features, failed = _load_checkpoint(manifest_path, features_path, failed_path)
    done_files = {str(m['file']) for m in manifest}
    if features and verbose:
        print(f"Resuming stellar_flare: {len(features)}/{target} already survived, "
              f"{len(failed)} known failures will be skipped")
    if len(features) >= target:
        return manifest[:target], features[:target]

    bar = progress(total=target, initial=len(features), desc='stellar_flare (survivors)') if progress else None
    stars_used = 0
    try:
        for name in star_names:
            if len(features) >= target:
                break
            stars_used += 1
            try:
                search = search_fn(name)
            except Exception:
                if verbose:
                    print(f"  search failed for {name}, moving on")
                continue
            if search is None or len(search) == 0:
                continue

            safe_name = str(name).replace(' ', '_')
            for i, entry in enumerate(list(search)[:max_sectors_per_star]):
                if len(features) >= target:
                    break
                fname = f"{safe_name}_sector{i}.csv"
                if fname in done_files or fname in failed:
                    continue
                try:
                    df = download_fn(entry)
                except Exception:
                    failed.add(fname)
                    continue
                if df is None or len(df) == 0:
                    failed.add(fname)
                    continue

                csv_path = os.path.join(out_dir, fname)
                df.to_csv(csv_path, index=False)

                feat = process_fn(csv_path, name, 'stellar_flare')
                if feat is None:
                    # Cleaning/GP failed -> next sector, then next star. Same retry contract.
                    failed.add(fname)
                    continue

                feat = dict(feat)
                # Keep a per-sector unique id so 14 sectors of AD Leo aren't 14 rows all
                # called "AD Leo" — they are distinct light curves and must stay distinguishable.
                feat['id'] = f"{safe_name}_sector{i}"
                manifest.append({'star_name': name, 'file': fname, 'label': 'stellar_flare',
                                 'n_points': len(df)})
                features.append(feat)
                done_files.add(fname)
                if bar is not None:
                    bar.update(1)

            if stars_used % checkpoint_every == 0:
                _save_checkpoint(manifest, features, failed,
                                 manifest_path, features_path, failed_path)
                gc.collect()
    finally:
        if bar is not None:
            bar.close()
        _save_checkpoint(manifest, features, failed, manifest_path, features_path, failed_path)

    if verbose:
        if len(features) < target:
            print(f"  SHORTFALL: stellar_flare reached only {len(features)}/{target} after working "
                  f"through {stars_used} stars ({len(failed)} cumulative failures). Add more names to "
                  f"FLARE_STAR_NAMES and/or raise max_sectors_per_star, then re-run; it resumes.")
        else:
            print(f"  stellar_flare: reached target {target}/{target} "
                  f"(used {stars_used} stars, {len(failed)} cumulative failures).")
    return manifest[:target], features[:target]


def assemble_feature_df(feature_lists, verbose=True):
    """
    Combine per-class feature rows into one frame, flag rows that genuinely had a
    two-band colour measurement, and impute the rest.

    `has_color` is kept as a feature precisely because "colour was measurable at
    all" is itself informative: TESS flares never have a ZTF g-r colour, so the
    flag is a real observational signal, not just an imputation artefact.
    """
    rows = []
    for lst in feature_lists:
        rows.extend(lst)
    df = pd.DataFrame(rows)
    df['has_color'] = df['color_g_r'].notna().astype(int)
    global_median = df.loc[df['has_color'] == 1, 'color_g_r'].median()
    class_medians = df.groupby('label')['color_g_r'].transform('median')
    df['color_g_r'] = df['color_g_r'].fillna(class_medians).fillna(global_median)
    if verbose:
        print(f"Combined feature table: {df.shape[0]} rows x {df.shape[1]} cols")
    return df.reset_index(drop=True)


def verify_counts(feature_df, n_per_class, n_per_subtype, sn_subtype_labels, verbose=True):
    """
    Definition-of-done check: 150 per coarse class, 30 per SN subtype.
    Returns (ok, report_df). Never raises — a shortfall is reported, not hidden.
    """
    coarse = feature_df['label'].apply(lambda l: 'SNe' if l in sn_subtype_labels else l)
    rows = []
    ok = True
    for cls in sorted(coarse.unique()):
        n = int((coarse == cls).sum())
        rows.append({'level': 'coarse', 'class': cls, 'count': n,
                     'target': n_per_class, 'shortfall': max(0, n_per_class - n)})
        ok = ok and n == n_per_class
    for st in sn_subtype_labels:
        n = int((feature_df['label'] == st).sum())
        rows.append({'level': 'subtype', 'class': st, 'count': n,
                     'target': n_per_subtype, 'shortfall': max(0, n_per_subtype - n)})
        ok = ok and n == n_per_subtype
    report = pd.DataFrame(rows)
    if verbose:
        print(report.to_string(index=False))
        print("All targets met." if ok else "NOT all targets met — see 'shortfall' column above.")
    return ok, report

In [ ]:
# Wire the generalised loops to the real APIs. Each adapter is a thin shim over the
# existing, unchanged functions — this is where dependency injection meets reality.
import lightkurve as lk

def _resolve_fn(row):
    return resolve_oid(row)

def _detections_fn(oid):
    return alerce.query_detections(oid, format='pandas', sort='mjd')

def _process_ztf_fn(csv_path, oid, label):
    return process_ztf_object(csv_path, oid, label)

def _flare_search_fn(star_name):
    return lk.search_lightcurve(star_name, mission='TESS', author='SPOC', exptime=120)

def _flare_download_fn(entry):
    lc = entry.download().remove_nans()
    df = lc.to_pandas().reset_index()[['time', 'flux', 'flux_err']]
    return df.rename(columns={'time': 'bjd'})

def _process_tess_fn(csv_path, star_name, label):
    return process_tess_object(csv_path, star_name, label)

# How deep a candidate pool to draw per class. Attrition through cross-match +
# download + cleaning + GP has historically run 40-60%, so ~8x the target leaves
# comfortable headroom; the loop stops as soon as the target is met, so an
# over-generous pool costs nothing but a bigger shuffle.
CANDIDATE_POOL_MULTIPLIER = 8

def _pool_for(mask, target, seed=42):
    pool = tns_full[mask][[c for c in KEEP_COLS if c in tns_full.columns]].copy()
    n = min(len(pool), target * CANDIDATE_POOL_MULTIPLIER)
    return pool.sample(n, random_state=seed).reset_index(drop=True)

# Label and feature vocabulary, defined once and used by both phases.
SN_SUBTYPE_LABELS_LIST = ['SN_Ia', 'SN_Ib', 'SN_Ic', 'SN_II', 'SLSN']
FEATURES = ['peak_val', 'rise_time', 'decay_time', 'amplitude', 'color_g_r']
X_COLS = FEATURES + ['has_color']

print(f"Targets: {N_PER_SN_SUBTYPE} per SN subtype x 5 = {5 * N_PER_SN_SUBTYPE} SNe, "
      f"{N_PER_CLASS} each for AGN / TDE / stellar_flare")

In [ ]:
# ---- The 5 SN subtypes + AGN + TDE, all through the same retry loop ----
tns_features, tns_manifests = {}, {}

for subtype in SN_SUBTYPES:
    pool = _pool_for(tns_full['sn_subtype'] == subtype, N_PER_SN_SUBTYPE)
    tns_manifests[subtype], tns_features[subtype] = build_tns_class_to_target(
        subtype, pool, N_PER_SN_SUBTYPE,
        base_dir=BASE_DIR, feature_dir=FEATURE_DIR,
        resolve_oid_fn=_resolve_fn, query_detections_fn=_detections_fn,
        process_fn=_process_ztf_fn, progress=tqdm)

for label, mask in [('AGN', agn_mask), ('TDE', tde_mask)]:
    pool = _pool_for(mask, N_PER_CLASS)
    tns_manifests[label], tns_features[label] = build_tns_class_to_target(
        label, pool, N_PER_CLASS,
        base_dir=BASE_DIR, feature_dir=FEATURE_DIR,
        resolve_oid_fn=_resolve_fn, query_detections_fn=_detections_fn,
        process_fn=_process_ztf_fn, progress=tqdm)

In [ ]:
# ---- Stellar flares, same contract, TESS-native path ----
flare_manifest_rows, flare_features = build_flares_to_target(
    FLARE_STAR_NAMES, N_PER_CLASS,
    base_dir=BASE_DIR, feature_dir=FEATURE_DIR,
    search_fn=_flare_search_fn, download_fn=_flare_download_fn,
    process_fn=_process_tess_fn, progress=tqdm)

for label in list(tns_features) + ['stellar_flare']:
    n = len(flare_features) if label == 'stellar_flare' else len(tns_features[label])
    target = N_PER_CLASS if label in ('AGN', 'TDE', 'stellar_flare') else N_PER_SN_SUBTYPE
    print(f"  {label:<16} {n}/{target}")

In [ ]:
# Persist the balanced manifests under the same names the previous notebook used, so
# a kernel restart can reload them and the sanity-check plots below have something to
# point at.
manifest_dfs = {label: pd.DataFrame(rows) for label, rows in tns_manifests.items()}
manifest_dfs['stellar_flare'] = pd.DataFrame(flare_manifest_rows)

for label, mdf in manifest_dfs.items():
    mdf.to_csv(os.path.join(BASE_DIR, f'{label.lower()}_manifest.csv'), index=False)

sn_ia_manifest = manifest_dfs['SN_Ia']
flare_manifest = manifest_dfs['stellar_flare']
print({k: len(v) for k, v in manifest_dfs.items()})

In [ ]:
# ---- Combine and verify against the definition of done ----
feature_df = assemble_feature_df(
    [tns_features[s] for s in SN_SUBTYPES] +
    [tns_features['AGN'], tns_features['TDE'], flare_features])

feature_df.to_csv(os.path.join(FEATURE_DIR, 'phase3_features_balanced.csv'), index=False)

counts_ok, count_report = verify_counts(
    feature_df, N_PER_CLASS, N_PER_SN_SUBTYPE, SN_SUBTYPE_LABELS_LIST)

In [ ]:
# GP sanity check
def plot_gp_sanity_check(oid_or_name, folder, file_col_is_oid=True, value_col='magpsf', is_mag=True):
    fname = f"{oid_or_name}.csv" if file_col_is_oid else oid_or_name
    path = os.path.join(BASE_DIR, folder, fname)
    if not os.path.exists(path):
        print(f"File not found: {path}")
        return
    df = pd.read_csv(path)
    if 'fid' in df.columns:
        # Filter to one band for visualization
        df = df[df['fid'] == 2]
    x_col = 'mjd' if 'mjd' in df.columns else 'bjd'
    clean_df = clean_lightcurve(df[[x_col, value_col]], value_col, min_points=6)
    if clean_df is None:
        print(f"skipped {oid_or_name}: too few points"); return
    t_grid, y_grid = gp_interpolate(clean_df[x_col].values, clean_df[value_col].values)
    plt.figure(figsize=(6, 3))
    plt.scatter(clean_df[x_col], clean_df[value_col], s=15, label='raw data')
    plt.plot(t_grid, y_grid, color='crimson', label='GP fit')
    if is_mag: plt.gca().invert_yaxis()
    plt.title(f"GP sanity check: {oid_or_name}"); plt.legend(); plt.show()
    plt.close('all')

In [ ]:
# Spot-check one ZTF light curve and one TESS light curve against their GP fits.
if len(sn_ia_manifest):
    plot_gp_sanity_check(sn_ia_manifest.iloc[0]['oid'], 'sn_ia')

if len(flare_manifest):
    plot_gp_sanity_check(flare_manifest.iloc[0]['file'], 'stellar_flares',
                         file_col_is_oid=False, value_col='flux', is_mag=False)

In [ ]:
# Feature distributions by class
FEATURES = ['peak_val', 'rise_time', 'decay_time', 'amplitude', 'color_g_r']

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, feat in zip(axes, FEATURES):
    for label in feature_df['label'].unique():
        subset = feature_df[feature_df['label'] == label][feat]
        ax.hist(subset, bins=20, alpha=0.5, label=label)
    ax.set_title(feat); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FEATURE_DIR, 'phase3_feature_distributions.png'), dpi=150)
plt.show()
plt.close('all')

# PHASE 3 — Hierarchical Classification

### The split, stated once

There is exactly one stratified 80/20 split, computed over the full 600-row
`feature_df` and stratified on the coarse label:

- **Stage 1** trains and evaluates on it directly.
- **Stage 2** trains only on the SN-labelled rows *inside Stage 1's training
  partition*, and evaluates only on the SN-labelled rows *inside Stage 1's test
  partition*.

An SN object in Stage 1's test set therefore cannot appear in Stage 2's training
set. `assert_split_nesting` checks this and fails loudly if it is ever violated.

In [ ]:
# Phase 3 — split, models, Stage 1 / Stage 2
# ---------------------------------------------------------------
# Inlined verbatim from btp_pipeline/modeling.py, which is covered by the
# repository's synthetic-data test suite. Edit it there and rebuild this
# notebook (tools/build_notebook.py) rather than editing it here.
# ---------------------------------------------------------------

"""
Phase 3 — hierarchical model training and evaluation.

Design correction A (the important one): there is exactly ONE stratified split,
computed over the full combined feature table and stratified on the coarse label.
Stage 1 trains and evaluates on it directly. Stage 2's training rows are the
SN-labelled rows *inside Stage 1's training partition*, and its evaluation rows
are the SN-labelled rows *inside Stage 1's test partition*. Stage 2 therefore
cannot see a Stage 1 test object during training — leakage is impossible by
construction, so no "leakage-free retrain" cleanup step is needed afterwards.
"""

import numpy as np
import pandas as pd
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier


# --------------------------------------------------------------------------
# Labels and the one global split
# --------------------------------------------------------------------------

def add_coarse_label(feature_df, sn_subtype_labels):
    """Collapse the 5 SN subtypes into a single 'SNe' coarse label, keeping
    the fine-grained subtype untouched in 'label'."""
    df = feature_df.copy()
    df['coarse_label'] = df['label'].apply(lambda l: 'SNe' if l in sn_subtype_labels else l)
    return df


def make_global_split(feature_df, x_cols, test_size=0.2, random_state=42, verbose=True):
    """
    The single source of truth for who is 'train' and who is 'test' anywhere in
    this project. Stratified on the coarse label so all four coarse classes are
    proportionally represented on both sides.

    Returns a dict carrying positional indices into `feature_df`; every later
    stage slices these rather than splitting again.
    """
    X = feature_df[x_cols].values
    le_coarse = LabelEncoder()
    y_coarse = le_coarse.fit_transform(feature_df['coarse_label'].values)

    idx_all = np.arange(len(feature_df))
    idx_train, idx_test = train_test_split(
        idx_all, test_size=test_size, stratify=y_coarse, random_state=random_state
    )
    idx_train.sort()
    idx_test.sort()

    split = {
        'X': X,
        'x_cols': list(x_cols),
        'idx_train': idx_train,
        'idx_test': idx_test,
        'y_coarse': y_coarse,
        'le_coarse': le_coarse,
        'feature_df': feature_df,
    }
    if verbose:
        print(f"Global split: {len(idx_train)} train / {len(idx_test)} test "
              f"(stratified on coarse_label, test_size={test_size})")
        print(pd.crosstab(feature_df['coarse_label'],
                          np.where(np.isin(idx_all, idx_train), 'train', 'test')))
    return split


def assert_split_nesting(split, sn_subtype_labels):
    """
    Guard the design correction. Fails loudly if Stage 2's rows are ever anything
    other than a subset of Stage 1's own train/test partitions.
    """
    df = split['feature_df']
    sn_mask = df['label'].isin(sn_subtype_labels).values
    s2_train = np.array([i for i in split['idx_train'] if sn_mask[i]])
    s2_test = np.array([i for i in split['idx_test'] if sn_mask[i]])

    assert set(s2_train).issubset(set(split['idx_train'])), "Stage 2 train escaped Stage 1 train"
    assert set(s2_test).issubset(set(split['idx_test'])), "Stage 2 test escaped Stage 1 test"
    assert not (set(s2_train) & set(split['idx_test'])), "LEAKAGE: Stage 2 trains on a Stage 1 test object"
    assert not (set(s2_test) & set(split['idx_train'])), "LEAKAGE: Stage 2 evaluates on a Stage 1 train object"
    return s2_train, s2_test


def safe_cv_folds(y_arr, desired=5, verbose=True):
    """
    Adaptive fold count (design correction C). Even at a clean 30 per subtype,
    an 80/20 split leaves ~24 per class in training and the split does not always
    land evenly, so a fixed cv=5 is not guaranteed safe. Degrade gracefully
    instead of crashing cross_val_score.
    """
    counts = pd.Series(y_arr).value_counts()
    min_count = int(counts.min())
    folds = min(desired, min_count)
    if folds < desired and verbose:
        print(f"NOTE: reducing CV folds {desired} -> {max(folds, 2)}; smallest class in this "
              f"training split has only {min_count} examples.")
    return max(folds, 2)


# --------------------------------------------------------------------------
# The 4 models
# --------------------------------------------------------------------------

def build_models(random_state=42, n_jobs=-1):
    """
    Fresh, unfitted instances of the four models, keyed by their canonical
    display names.

    Bagging (SVM) deliberately does NOT copy the tree-bagging configuration
    (design correction D):
      * n_estimators=50 rather than 300. Each base SVC is O(n^2)-ish to fit and
        there is no cheap warm start, so 300 bagged SVCs is minutes of compute
        for no measurable accuracy gain at n=600.
      * probability left at its default (off). Enabling it triggers an internal
        5-fold Platt-scaling CV *inside every base estimator* — 50x (or 300x) a
        5-fold refit. It is not passed explicitly because scikit-learn 1.9
        deprecated the keyword; the default is already what we want. Nothing
        downstream needs calibrated probabilities from this model: the confusion
        matrices, the cascade and the decision-boundary plots all call .predict,
        and feature importance comes from permutation importance, which also only
        needs .predict. If calibrated probabilities are ever needed, flip it on
        for that analysis alone.
      * n_jobs=-1 to parallelise across base estimators.
    """
    return {
        'Random Forest': RandomForestClassifier(
            n_estimators=300, class_weight='balanced', random_state=random_state, n_jobs=n_jobs),
        'Logistic Regression': LogisticRegression(
            max_iter=1000, class_weight='balanced', random_state=random_state),
        'Bagging (Trees)': BaggingClassifier(
            estimator=DecisionTreeClassifier(class_weight='balanced', random_state=random_state),
            n_estimators=300, max_samples=0.8, max_features=0.7,
            random_state=random_state, n_jobs=n_jobs),
        'Bagging (SVM)': BaggingClassifier(
            estimator=SVC(kernel='rbf', class_weight='balanced', random_state=random_state),
            n_estimators=50, max_samples=0.8, max_features=0.7,
            random_state=random_state, n_jobs=n_jobs),
    }


def _scale(X_train, X_test):
    scaler = StandardScaler()
    return scaler, scaler.fit_transform(X_train), scaler.transform(X_test)


def train_and_evaluate(X_train, y_train, X_test, y_test, le, stage_name,
                       random_state=42, cv_desired=5, verbose=True):
    """
    Fit all four models on one (already-scaled) split and score them.

    Returns (fitted_models, results_df, confusions) where results_df carries CV
    mean/std on the training partition and accuracy on the held-out test rows.
    """
    models = build_models(random_state=random_state)
    folds = safe_cv_folds(y_train, desired=cv_desired, verbose=verbose)

    fitted, rows, confusions = {}, [], {}
    labels_idx = np.arange(len(le.classes_))
    for name, model in models.items():
        cv = cross_val_score(model, X_train, y_train, cv=folds, scoring='accuracy')
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        fitted[name] = model
        confusions[name] = confusion_matrix(y_test, y_pred, labels=labels_idx)
        rows.append({'Model': name, 'CV folds': folds, 'CV accuracy': cv.mean(),
                     'CV std': cv.std(), 'Test accuracy': acc})
        if verbose:
            print(f"[{stage_name}] {name:<22} CV {cv.mean():.3f} +/- {cv.std():.3f} | "
                  f"test {acc:.3f}")
    return fitted, pd.DataFrame(rows), confusions


# --------------------------------------------------------------------------
# Stage 1 / Stage 2
# --------------------------------------------------------------------------

def run_stage1(split, random_state=42, verbose=True):
    """Coarse classification: SNe / AGN / TDE / stellar_flare, on the global split."""
    X, idx_train, idx_test = split['X'], split['idx_train'], split['idx_test']
    y = split['y_coarse']
    le = split['le_coarse']

    scaler, X_train_s, X_test_s = _scale(X[idx_train], X[idx_test])
    y_train, y_test = y[idx_train], y[idx_test]

    fitted, results, confusions = train_and_evaluate(
        X_train_s, y_train, X_test_s, y_test, le, 'Stage 1',
        random_state=random_state, verbose=verbose)

    return {'stage': 'Stage 1', 'scaler': scaler, 'le': le, 'models': fitted,
            'results': results, 'confusions': confusions,
            'X_train': X_train_s, 'X_test': X_test_s,
            'y_train': y_train, 'y_test': y_test,
            'idx_train': idx_train, 'idx_test': idx_test}


def run_stage2_option_a(split, sn_subtype_labels, random_state=42, verbose=True):
    """
    Option A — the 'ceiling'. Subtype the TRUE SN rows directly, i.e. perfect
    ground-truth routing with no Stage 1 errors in the way. Train rows are the
    SN rows of Stage 1's training partition; test rows are the SN rows of Stage
    1's test partition. Nothing is re-split.
    """
    df, X = split['feature_df'], split['X']
    s2_train, s2_test = assert_split_nesting(split, sn_subtype_labels)

    le = LabelEncoder()
    le.fit(df['label'].iloc[np.concatenate([s2_train, s2_test])].values)
    y_train = le.transform(df['label'].iloc[s2_train].values)
    y_test = le.transform(df['label'].iloc[s2_test].values)

    scaler, X_train_s, X_test_s = _scale(X[s2_train], X[s2_test])
    if verbose:
        print(f"Stage 2 (Option A): {len(s2_train)} train / {len(s2_test)} test true-SN rows")
        print(pd.Series(df['label'].iloc[s2_train].values).value_counts().to_string())

    fitted, results, confusions = train_and_evaluate(
        X_train_s, y_train, X_test_s, y_test, le, 'Stage 2 / A',
        random_state=random_state, verbose=verbose)

    return {'stage': 'Stage 2 Option A', 'scaler': scaler, 'le': le, 'models': fitted,
            'results': results, 'confusions': confusions,
            'X_train': X_train_s, 'X_test': X_test_s,
            'y_train': y_train, 'y_test': y_test,
            'idx_train': s2_train, 'idx_test': s2_test}


def cascade_predict(stage1_model, stage2_model, X_raw, scaler1, scaler2, le1, le2,
                    sn_coarse_label='SNe'):
    """
    Stage 1 decides SNe vs not. Only objects Stage 1 *predicted* to be SNe get a
    Stage 2 subtype; everything else keeps Stage 1's coarse answer as final.
    """
    coarse_pred = le1.inverse_transform(stage1_model.predict(scaler1.transform(X_raw)))
    final = np.array(coarse_pred, dtype=object)
    sne_mask = coarse_pred == sn_coarse_label
    if sne_mask.any():
        fine = le2.inverse_transform(stage2_model.predict(scaler2.transform(X_raw[sne_mask])))
        final[sne_mask] = fine
    return {'coarse_pred': coarse_pred, 'final_pred': final, 'sne_mask': sne_mask}


def run_stage2_option_b(split, stage1, stage2a, sn_subtype_labels, verbose=True):
    """
    Option B — the 'realistic' path. Subtype whatever Stage 1 *predicted* was an
    SN, including Stage 1's mistakes. Reuses the Option A models: they were
    trained on true SN rows from Stage 1's training partition, which is the only
    subtype-labelled training data that exists. What differs is the population
    they are applied to at test time.

    Three metrics, because they answer three different questions:
      * End-to-end accuracy — final 8-class prediction vs true fine label, over
        ALL test objects. The honest "what does the whole pipeline do" number,
        but flattered by the easy non-SN classes.
      * Conditional accuracy — subtype accuracy among true SNe that Stage 1
        actually routed correctly. Isolates Stage 2's own skill.
      * Option-A-comparable accuracy — over true SN test rows only, counting a
        Stage 1 mis-route as wrong. This is the only number directly comparable
        to Option A, because it is the same population scored the same way.
    """
    df, X = split['feature_df'], split['X']
    idx_test = split['idx_test']
    X_test_raw = X[idx_test]
    true_fine = df['label'].iloc[idx_test].values
    true_coarse = df['coarse_label'].iloc[idx_test].values
    is_true_sn = np.isin(true_fine, sn_subtype_labels)

    results, rows = {}, []
    for name in stage1['models']:
        r = cascade_predict(stage1['models'][name], stage2a['models'][name], X_test_raw,
                            stage1['scaler'], stage2a['scaler'],
                            stage1['le'], stage2a['le'])
        final, coarse_pred, sne_mask = r['final_pred'], r['coarse_pred'], r['sne_mask']

        end_to_end = float((final == true_fine).mean())

        routed_ok = is_true_sn & (coarse_pred == 'SNe')
        conditional = float((final[routed_ok] == true_fine[routed_ok]).mean()) if routed_ok.any() else np.nan

        comparable = float((final[is_true_sn] == true_fine[is_true_sn]).mean()) if is_true_sn.any() else np.nan

        fp_leak = int((~is_true_sn & sne_mask).sum())     # non-SNe wrongly sent into Stage 2
        fn_loss = int((is_true_sn & ~sne_mask).sum())     # true SNe Stage 1 never routed

        r.update({'true_fine': true_fine, 'true_coarse': true_coarse, 'is_true_sn': is_true_sn})
        results[name] = r
        rows.append({'Model': name,
                     'End-to-end accuracy (all test objects)': end_to_end,
                     'Conditional accuracy (correctly-routed SNe)': conditional,
                     'Option-A-comparable accuracy (true SNe)': comparable,
                     'Non-SNe mis-routed into Stage 2': fp_leak,
                     'True SNe lost before Stage 2': fn_loss})
        if verbose:
            print(f"[Stage 2 / B] {name:<22} end-to-end {end_to_end:.3f} | "
                  f"conditional {conditional:.3f} | comparable {comparable:.3f} | "
                  f"FP-in {fp_leak}, FN-lost {fn_loss}")

    summary = pd.DataFrame(rows)
    all_classes = sorted(pd.unique(df['label']))
    confusions = {name: confusion_matrix(true_fine, results[name]['final_pred'], labels=all_classes)
                  for name in results}
    return {'stage': 'Stage 2 Option B', 'per_model': results, 'summary': summary,
            'confusions': confusions, 'all_classes': all_classes, 'idx_test': idx_test}


def option_ab_comparison(stage2a, stage2b):
    """Side-by-side Option A vs Option B, per model, on matched populations."""
    a = stage2a['results'].set_index('Model')['Test accuracy']
    b = stage2b['summary'].set_index('Model')
    out = pd.DataFrame({
        'Option A (ground-truth routing)': a,
        'Option B (comparable: true SNe, mis-route = wrong)':
            b['Option-A-comparable accuracy (true SNe)'],
        'Option B (conditional: correctly-routed only)':
            b['Conditional accuracy (correctly-routed SNe)'],
        'Option B (end-to-end, all 8 classes)':
            b['End-to-end accuracy (all test objects)'],
    })
    out['Routing cost (A - B comparable)'] = (
        out['Option A (ground-truth routing)']
        - out['Option B (comparable: true SNe, mis-route = wrong)'])
    return out.reset_index()


def repeated_cv_estimate(X_train, y_train, le, stage_name, n_splits=5, n_repeats=10,
                         random_state=42, verbose=True):
    """
    A better-resolved accuracy estimate than a single small held-out set.

    Why this exists. Stage 2's held-out test set is the SN rows inside Stage 1's
    20% test partition: 30 objects, ~6 per subtype. A single accuracy on 30 draws
    has a binomial standard error of ~9 percentage points, and a per-subtype
    recall computed on 6 objects moves in steps of 17 points. That resolution is
    coarser than the effect the rebuild is trying to measure, so the single-split
    number cannot by itself tell you whether Stage 2 improved.

    Repeated stratified K-fold over the Stage 2 TRAINING partition reuses every
    training object as a validation object across repeats, giving a mean and a
    spread. It never touches Stage 1's test partition, so it introduces no
    leakage. Report it alongside — not instead of — the held-out number: the
    held-out set remains the only fully untouched evaluation.
    """
    from sklearn.model_selection import RepeatedStratifiedKFold

    n_splits = int(min(n_splits, pd.Series(y_train).value_counts().min()))
    n_splits = max(n_splits, 2)
    cv = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats,
                                 random_state=random_state)
    rows = []
    for name, model in build_models(random_state=random_state).items():
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
        lo, hi = np.percentile(scores, [2.5, 97.5])
        rows.append({'Model': name, 'Repeated-CV mean': scores.mean(),
                     'Repeated-CV std': scores.std(),
                     'CI 2.5%': lo, 'CI 97.5%': hi,
                     'n_fits': len(scores)})
        if verbose:
            print(f"[{stage_name}] {name:<22} repeated CV {scores.mean():.3f} "
                  f"+/- {scores.std():.3f}  (95% range {lo:.3f}-{hi:.3f}, {len(scores)} fits)")
    return pd.DataFrame(rows)


def single_split_resolution(n_test, n_classes):
    """
    The honest error bar on an accuracy measured from `n_test` objects, plus how
    coarse a per-class recall is at this sample size. Printed next to every
    headline number so nobody over-reads a 30-object result.
    """
    per_class = n_test / n_classes if n_classes else np.nan
    return {
        'n_test': n_test,
        'worst_case_std_error': float(0.5 / np.sqrt(n_test)) if n_test else np.nan,
        'accuracy_step': float(1.0 / n_test) if n_test else np.nan,
        'mean_test_objects_per_class': float(per_class),
        'per_class_recall_step': float(1.0 / per_class) if per_class else np.nan,
    }

In [ ]:
feature_df = add_coarse_label(feature_df, SN_SUBTYPE_LABELS_LIST)
print(feature_df['coarse_label'].value_counts().to_string())

split = make_global_split(feature_df, X_COLS, test_size=0.2, random_state=42)

# The guard. If this ever fires, the split has been recomputed somewhere it should not be.
s2_train_idx, s2_test_idx = assert_split_nesting(split, SN_SUBTYPE_LABELS_LIST)
print(f"\nStage 2 inherits {len(s2_train_idx)} train / {len(s2_test_idx)} test rows "
      f"from Stage 1's own partitions.")
print("Leakage check PASSED — Stage 2 rows are subsets of Stage 1's by construction.")

### Stage 1 — coarse classification (SNe / AGN / TDE / Stellar Flare)

In [ ]:
stage1 = run_stage1(split, random_state=42)
print()
display(stage1['results'].round(3))

### Stage 2 — Option A: ground-truth routing (the ceiling)

Subtype the **true** SN objects directly, with no Stage 1 errors in the way. This is
the upper bound on what Stage 2 can do; it is not achievable in deployment, because
in deployment nothing tells you which objects are really SNe.

In [ ]:
stage2a = run_stage2_option_a(split, SN_SUBTYPE_LABELS_LIST, random_state=42)
print()
display(stage2a['results'].round(3))

### Stage 2 — Option B: Stage-1-predicted SNe (the realistic path)

Subtype whatever **Stage 1 predicted** was an SN — including Stage 1's mistakes.
Three numbers, because they answer three different questions:

| metric | population | what it tells you |
| --- | --- | --- |
| End-to-end accuracy | all test objects, 8 classes | what the whole pipeline does — but flattered by the easy non-SN classes |
| Conditional accuracy | true SNe that Stage 1 routed correctly | Stage 2's own skill, isolated from routing |
| Option-A-comparable | true SN test rows, mis-route counted wrong | the **only** number directly comparable to Option A |

In [ ]:
stage2b = run_stage2_option_b(split, stage1, stage2a, SN_SUBTYPE_LABELS_LIST)
print()
display(stage2b['summary'].round(3))

### Option A vs Option B — side by side

In [ ]:
comparison = option_ab_comparison(stage2a, stage2b)
display(comparison.round(3))

print("\nThe honest comparison is Option A vs 'Option B comparable': same population,")
print("same scoring, the only difference being that Option B must survive Stage 1 routing.")
print("'Routing cost' is the price of the hierarchy, per model.")
comparison.to_csv(os.path.join(FEATURE_DIR, 'option_ab_comparison.csv'), index=False)

### How much of this is signal? — measurement resolution

Balancing the dataset removed the *bias* caused by uneven attrition. It could not
remove the *variance*: 30 objects per subtype is a small sample, and after an 80/20
split Stage 2's held-out set is about 30 objects in total, ~6 per subtype.

The cell below states the error bar explicitly, and adds a repeated stratified-CV
estimate over the Stage 2 **training** partition — many more fits, and still no
contact with Stage 1's test partition, so it adds no leakage. Read the held-out
number as the untouched evaluation and the repeated-CV number when judging whether
two models genuinely differ.

In [ ]:
res_s1 = single_split_resolution(len(stage1['y_test']), len(stage1['le'].classes_))
res_s2 = single_split_resolution(len(stage2a['y_test']), len(stage2a['le'].classes_))
for tag, r in [('Stage 1', res_s1), ('Stage 2', res_s2)]:
    print(f"{tag}: n_test={r['n_test']}, accuracy resolution {r['accuracy_step']:.3f}, "
          f"worst-case SE {r['worst_case_std_error']:.3f}, "
          f"~{r['mean_test_objects_per_class']:.1f} objects/class "
          f"(per-class recall moves in steps of {r['per_class_recall_step']:.2f})")

print()
repeated_cv_s2 = repeated_cv_estimate(stage2a['X_train'], stage2a['y_train'],
                                      stage2a['le'], 'Stage 2 / A',
                                      n_splits=5, n_repeats=10)
display(repeated_cv_s2.round(3))

In [ ]:
# Persist everything needed to reproduce or write up the results later.
import joblib
for tag, st in [('stage1', stage1), ('stage2a', stage2a)]:
    for name, model in st['models'].items():
        safe = name.replace(' ', '_').replace('(', '').replace(')', '')
        joblib.dump(model, os.path.join(FEATURE_DIR, f'{tag}_{safe}.pkl'))
    joblib.dump(st['scaler'], os.path.join(FEATURE_DIR, f'{tag}_scaler.pkl'))
    joblib.dump(st['le'], os.path.join(FEATURE_DIR, f'{tag}_label_encoder.pkl'))
np.save(os.path.join(FEATURE_DIR, 'split_idx_train.npy'), split['idx_train'])
np.save(os.path.join(FEATURE_DIR, 'split_idx_test.npy'), split['idx_test'])

for name, r in stage2b['per_model'].items():
    safe = name.replace(' ', '_').replace('(', '').replace(')', '')
    out = feature_df.iloc[split['idx_test']][['id', 'label', 'coarse_label']].copy()
    out['stage1_pred'] = r['coarse_pred']
    out['final_pred'] = r['final_pred']
    out['routed_to_stage2'] = r['sne_mask']
    out.to_csv(os.path.join(FEATURE_DIR, f'cascade_predictions_{safe}.csv'), index=False)
print('Saved models, split indices and cascade predictions to', FEATURE_DIR)

# PHASE 4 — Interpretation and Physical Insights

Applied to **both** stages, per the project brief: feature importance, the most
discriminative features, misclassification analysis, confusion matrices, decision
boundaries, and the physics connection.

**On importance methods.** Random Forest offers Gini importance, Logistic Regression
offers coefficients, Bagging (Trees) can average importances over its estimators —
and a bagged RBF SVM offers nothing at all. Those three are also not on comparable
scales. So **permutation importance on the held-out test set** is computed
identically for all four models, and native importances are reported alongside it
where they exist, as a cross-check rather than as the comparison.

In [ ]:
# Phase 4 — interpretation
# ---------------------------------------------------------------
# Inlined verbatim from btp_pipeline/interpret.py, which is covered by the
# repository's synthetic-data test suite. Edit it there and rebuild this
# notebook (tools/build_notebook.py) rather than editing it here.
# ---------------------------------------------------------------

"""
Phase 4 — interpretation and physical insight, applied to BOTH stages.

Covers the brief's Phase 4 deliverables: feature importance, most discriminative
features, misclassification analysis, confusion matrices, decision boundaries,
and the physics connection.

Design correction E: permutation importance is computed uniformly for all four
models on the held-out test set. Native importances (RF Gini, LR coefficients,
mean Gini across bagged trees) are reported alongside where they exist, but they
are not comparable across model families and an RBF-kernel bagged SVM has none
at all — so permutation importance is the one method that puts all four on the
same axis.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
from sklearn.metrics import ConfusionMatrixDisplay


# --------------------------------------------------------------------------
# Feature importance
# --------------------------------------------------------------------------

def permutation_importances(models, X_test, y_test, x_cols, n_repeats=20,
                            random_state=42, n_jobs=None):
    """
    Permutation importance for every model, on the held-out test set.
    Returns a tidy frame: one row per (model, feature) with mean and std.
    """
    rows = []
    for name, model in models.items():
        r = permutation_importance(model, X_test, y_test, n_repeats=n_repeats,
                                   random_state=random_state, scoring='accuracy',
                                   n_jobs=n_jobs)
        for feat, mean, std in zip(x_cols, r.importances_mean, r.importances_std):
            rows.append({'Model': name, 'feature': feat,
                         'perm_importance': float(mean), 'perm_std': float(std)})
    return pd.DataFrame(rows)


def native_importances(models, x_cols, le=None):
    """
    Each family's own importance measure, where one exists:
      Random Forest    -> Gini importance
      Bagging (Trees)  -> mean Gini across the bagged trees (features are
                          subsampled per estimator, so importances are mapped
                          back onto the full feature vector before averaging)
      LogReg           -> mean |standardised coefficient| across classes
      Bagging (SVM)    -> none; RBF SVMs expose neither coefficients nor
                          impurity gains. This is exactly why permutation
                          importance is the cross-family method.
    """
    rows = []
    for name, model in models.items():
        if hasattr(model, 'feature_importances_'):
            vals = np.asarray(model.feature_importances_, dtype=float)
            kind = 'Gini importance'
        elif hasattr(model, 'coef_'):
            vals = np.abs(np.atleast_2d(model.coef_)).mean(axis=0)
            kind = 'mean |standardised coef|'
        elif hasattr(model, 'estimators_') and hasattr(model.estimators_[0], 'feature_importances_'):
            acc = np.zeros(len(x_cols), dtype=float)
            cnt = np.zeros(len(x_cols), dtype=float)
            feats_per_est = getattr(model, 'estimators_features_', None)
            for est, feats in zip(model.estimators_, feats_per_est):
                acc[list(feats)] += est.feature_importances_
                cnt[list(feats)] += 1
            vals = np.divide(acc, np.maximum(cnt, 1))
            kind = 'mean Gini across bagged trees'
        else:
            continue
        for feat, v in zip(x_cols, vals):
            rows.append({'Model': name, 'feature': feat,
                         'native_importance': float(v), 'native_kind': kind})
    return pd.DataFrame(rows)


def plot_permutation_importance(perm_df, title, out_path=None):
    """One horizontal-bar panel per model, shared feature ordering."""
    models = list(dict.fromkeys(perm_df['Model']))
    order = (perm_df.groupby('feature')['perm_importance'].mean()
             .sort_values(ascending=False).index.tolist())
    fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, name in zip(axes, models):
        sub = perm_df[perm_df['Model'] == name].set_index('feature').loc[order]
        ax.barh(sub.index, sub['perm_importance'], xerr=sub['perm_std'],
                color='seagreen', alpha=0.85)
        ax.invert_yaxis()
        ax.axvline(0, color='0.4', lw=0.8)
        ax.set_title(name, fontsize=11)
        ax.set_xlabel('Δ accuracy when shuffled')
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=140, bbox_inches='tight')
    return fig


def rank_discriminative_features(perm_df):
    """Mean permutation importance across the four models — the headline
    'most discriminative features' answer for the brief."""
    return (perm_df.groupby('feature')['perm_importance']
            .agg(['mean', 'std']).sort_values('mean', ascending=False)
            .rename(columns={'mean': 'mean_perm_importance', 'std': 'across_model_std'}))


# --------------------------------------------------------------------------
# Confusion matrices
# --------------------------------------------------------------------------

def plot_confusions(confusions, class_names, title, accuracies=None, out_path=None):
    """A row of confusion matrices, one per model."""
    names = list(confusions)
    size = max(4.5, len(class_names) * 0.95)
    fig, axes = plt.subplots(1, len(names), figsize=(size * len(names), size))
    axes = np.atleast_1d(axes)
    for ax, name in zip(axes, names):
        disp = ConfusionMatrixDisplay(confusions[name], display_labels=class_names)
        disp.plot(ax=ax, cmap='Blues', colorbar=False, xticks_rotation=45, values_format='d')
        sub = f"{name}"
        if accuracies is not None and name in accuracies:
            sub += f"  (acc {accuracies[name]:.3f})"
        ax.set_title(sub, fontsize=11)
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=140, bbox_inches='tight')
    return fig


# --------------------------------------------------------------------------
# Misclassification analysis
# --------------------------------------------------------------------------

def misclassification_table(y_true_labels, y_pred_labels, top_n=10):
    """Which true->predicted confusions actually dominate the error budget."""
    err = pd.DataFrame({'true': y_true_labels, 'pred': y_pred_labels})
    err = err[err['true'] != err['pred']]
    if err.empty:
        return pd.DataFrame(columns=['true', 'pred', 'n', 'share_of_errors'])
    tab = (err.groupby(['true', 'pred']).size().reset_index(name='n')
           .sort_values('n', ascending=False))
    tab['share_of_errors'] = tab['n'] / tab['n'].sum()
    return tab.head(top_n).reset_index(drop=True)


def per_class_recall(y_true_labels, y_pred_labels):
    df = pd.DataFrame({'true': y_true_labels, 'pred': y_pred_labels})
    return (df.assign(correct=df['true'] == df['pred'])
            .groupby('true')['correct'].agg(['mean', 'size'])
            .rename(columns={'mean': 'recall', 'size': 'n_test'})
            .sort_values('recall'))


# --------------------------------------------------------------------------
# Decision boundaries (2D projection)
# --------------------------------------------------------------------------

def plot_decision_boundaries(feature_df, idx_train, idx_test, label_col, two_features,
                             build_models_fn, title, out_path=None, grid_steps=250,
                             random_state=42):
    """
    2D decision-boundary panels, one per model, on the two most discriminative
    features (we have 6, so this is a projection, not the real boundary).

    The models here are refit on those two features using the TRAINING rows only,
    and the TEST rows are scattered on top. Fitting the visualisation model on the
    whole dataset — as an earlier version of this notebook did — draws a boundary
    that has already seen the points it is being judged against, which makes the
    picture look tidier than the classifier actually is.
    """
    from sklearn.preprocessing import LabelEncoder, StandardScaler

    f1, f2 = two_features
    X2 = feature_df[[f1, f2]].values.astype(float)
    le = LabelEncoder()
    y2 = le.fit_transform(feature_df[label_col].values)

    scaler = StandardScaler()
    X2_train = scaler.fit_transform(X2[idx_train])
    X2_test = scaler.transform(X2[idx_test])
    y_train, y_test = y2[idx_train], y2[idx_test]

    pad = 0.6
    x_min, x_max = X2_train[:, 0].min() - pad, X2_train[:, 0].max() + pad
    y_min, y_max = X2_train[:, 1].min() - pad, X2_train[:, 1].max() + pad
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, grid_steps),
                         np.linspace(y_min, y_max, grid_steps))
    grid = np.c_[xx.ravel(), yy.ravel()]

    models = build_models_fn(random_state=random_state)
    fig, axes = plt.subplots(1, len(models), figsize=(5.2 * len(models), 4.8))
    axes = np.atleast_1d(axes)
    cmap = plt.get_cmap('tab10')
    for ax, (name, model) in zip(axes, models.items()):
        model.fit(X2_train, y_train)
        zz = model.predict(grid).reshape(xx.shape)
        ax.contourf(xx, yy, zz, alpha=0.25, levels=np.arange(-0.5, len(le.classes_)), cmap='tab10')
        for k, cls in enumerate(le.classes_):
            m = y_test == k
            ax.scatter(X2_test[m, 0], X2_test[m, 1], s=26, color=cmap(k % 10),
                       edgecolor='k', linewidth=0.4, label=cls)
        acc = float((model.predict(X2_test) == y_test).mean())
        ax.set_title(f"{name}  (2-feature test acc {acc:.2f})", fontsize=10)
        ax.set_xlabel(f'{f1} (standardised)')
        ax.set_ylabel(f'{f2} (standardised)')
    axes[0].legend(fontsize=8, loc='best')
    fig.suptitle(f"{title} — projection onto {f1} vs {f2}; points are held-out test objects",
                 fontsize=12)
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=140, bbox_inches='tight')
    return fig


# --------------------------------------------------------------------------
# Physics summary
# --------------------------------------------------------------------------

def physics_summary(feature_df, label_col, features):
    """Per-class mean/std of every physical feature — the table the write-up
    has to be consistent with."""
    return feature_df.groupby(label_col)[features].agg(['mean', 'std']).round(3)


def class_separation_ranking(feature_df, label_col, features):
    """
    A crude but useful 'how separable is this feature' score: between-class
    variance of the class means divided by the mean within-class variance
    (a one-way F-statistic in spirit). Complements permutation importance,
    which is model-mediated, with something purely about the data.
    """
    rows = []
    for f in features:
        g = feature_df.groupby(label_col)[f]
        between = g.mean().var(ddof=1)
        within = g.var(ddof=1).mean()
        rows.append({'feature': f, 'between_class_var': float(between),
                     'mean_within_class_var': float(within),
                     'separation_ratio': float(between / within) if within else np.nan})
    return pd.DataFrame(rows).sort_values('separation_ratio', ascending=False).reset_index(drop=True)

### Stage 1 interpretation

In [ ]:
perm_s1 = permutation_importances(stage1['models'], stage1['X_test'], stage1['y_test'],
                                  X_COLS, n_repeats=30)
display(perm_s1.pivot(index='feature', columns='Model', values='perm_importance').round(4))

print('Most discriminative features, averaged across the 4 models:')
ranked_s1 = rank_discriminative_features(perm_s1)
display(ranked_s1.round(4))

plot_permutation_importance(perm_s1, 'Stage 1 — permutation importance (held-out test set)',
                            os.path.join(FEATURE_DIR, 'stage1_perm_importance.png'))
plt.show()

In [ ]:
native_s1 = native_importances(stage1['models'], X_COLS)
display(native_s1.pivot(index='feature', columns='Model', values='native_importance').round(4))
print("Bagging (SVM) is absent above by design: an RBF-kernel SVM exposes neither")
print("coefficients nor impurity gains. That is exactly why permutation importance is")
print("the method used for the cross-model comparison.")

In [ ]:
plot_confusions(stage1['confusions'], list(stage1['le'].classes_),
                'Stage 1 — confusion matrices (held-out test set)',
                dict(zip(stage1['results']['Model'], stage1['results']['Test accuracy'])),
                os.path.join(FEATURE_DIR, 'stage1_confusion.png'))
plt.show()

yt_s1 = stage1['le'].inverse_transform(stage1['y_test'])
yp_s1 = stage1['le'].inverse_transform(stage1['models']['Random Forest'].predict(stage1['X_test']))
print('Per-class recall (Random Forest):')
display(per_class_recall(yt_s1, yp_s1).round(3))
print('Dominant confusions:')
misclass_s1 = misclassification_table(yt_s1, yp_s1)
display(misclass_s1)
recall_s1 = per_class_recall(yt_s1, yp_s1)

In [ ]:
top2_s1 = ranked_s1.index[:2].tolist()
print('Stage 1 decision boundary projected onto:', top2_s1)
plot_decision_boundaries(feature_df, split['idx_train'], split['idx_test'], 'coarse_label',
                         top2_s1, build_models, 'Stage 1',
                         os.path.join(FEATURE_DIR, 'stage1_decision_boundary.png'))
plt.show()
print("Note: the visualisation models are refit on these two features using TRAINING rows")
print("only, and the scattered points are held-out TEST objects. Fitting the visualisation")
print("on the full dataset — as an earlier version did — draws a boundary that has already")
print("seen the points it is judged against.")

### Stage 2 interpretation

In [ ]:
perm_s2 = permutation_importances(stage2a['models'], stage2a['X_test'], stage2a['y_test'],
                                  X_COLS, n_repeats=30)
ranked_s2 = rank_discriminative_features(perm_s2)
display(ranked_s2.round(4))
plot_permutation_importance(perm_s2, 'Stage 2 — permutation importance (held-out test set)',
                            os.path.join(FEATURE_DIR, 'stage2_perm_importance.png'))
plt.show()

plot_confusions(stage2a['confusions'], list(stage2a['le'].classes_),
                'Stage 2 Option A — confusion matrices (true SN test rows)',
                dict(zip(stage2a['results']['Model'], stage2a['results']['Test accuracy'])),
                os.path.join(FEATURE_DIR, 'stage2a_confusion.png'))
plt.show()

plot_confusions(stage2b['confusions'], stage2b['all_classes'],
                'Stage 2 Option B — cascaded, all original fine-grained classes',
                None, os.path.join(FEATURE_DIR, 'stage2b_confusion.png'))
plt.show()

In [ ]:
yt_s2 = stage2a['le'].inverse_transform(stage2a['y_test'])
yp_s2 = stage2a['le'].inverse_transform(stage2a['models']['Random Forest'].predict(stage2a['X_test']))
recall_s2 = per_class_recall(yt_s2, yp_s2)
misclass_s2 = misclassification_table(yt_s2, yp_s2)
print('Per-class recall (Random Forest, Option A):')
display(recall_s2.round(3))
print('Dominant subtype confusions:')
display(misclass_s2)
print(f"\nRead these with the sample size attached: {res_s2['n_test']} test objects total,")
print(f"~{res_s2['mean_test_objects_per_class']:.0f} per subtype. A recall of 0.00 or 1.00 here")
print("is not evidence of anything.")

In [ ]:
sn_rows = np.flatnonzero(feature_df['label'].isin(SN_SUBTYPE_LABELS_LIST).values)
sn_df = feature_df.iloc[sn_rows].reset_index(drop=True)
sn_positions = {orig: new for new, orig in enumerate(sn_rows)}
sn_train_local = np.array([sn_positions[i] for i in s2_train_idx])
sn_test_local = np.array([sn_positions[i] for i in s2_test_idx])

top2_s2 = ranked_s2.index[:2].tolist()
print('Stage 2 decision boundary projected onto:', top2_s2)
plot_decision_boundaries(sn_df, sn_train_local, sn_test_local, 'label', top2_s2,
                         build_models, 'Stage 2',
                         os.path.join(FEATURE_DIR, 'stage2_decision_boundary.png'))
plt.show()

In [ ]:
# Data-side separability — independent of any model — and the physics table.
print('Feature separability, Stage 1 (between-class variance / within-class variance):')
display(class_separation_ranking(feature_df, 'coarse_label', FEATURES).round(3))
print('Feature separability, Stage 2 (SN subtypes only):')
display(class_separation_ranking(sn_df, 'label', FEATURES).round(3))

phys_s1 = physics_summary(feature_df, 'coarse_label', FEATURES)
phys_s2 = physics_summary(sn_df, 'label', FEATURES)
phys_s1.to_csv(os.path.join(FEATURE_DIR, 'stage1_physics_summary.csv'))
phys_s2.to_csv(os.path.join(FEATURE_DIR, 'stage2_physics_summary.csv'))
display(phys_s1)
display(phys_s2)

# Results summary

The cell below generates the summary **from the result objects computed above** —
counts, accuracies, the Option A/B gap, the permutation-importance ranking and the
dominant confusions — and attaches the physics interpretation to whatever actually
came out on top. Nothing in it is typed in by hand, so it cannot disagree with the
tables it sits under.

In [ ]:
# Results summary generator
# ---------------------------------------------------------------
# Inlined verbatim from btp_pipeline/summary.py, which is covered by the
# repository's synthetic-data test suite. Edit it there and rebuild this
# notebook (tools/build_notebook.py) rather than editing it here.
# ---------------------------------------------------------------

"""
Auto-generated results summary.

The brief asks for a physics-grounded interpretation "tied to the actual numbers
obtained, not generic filler". So the write-up is assembled *from* the results
objects: which features actually ranked top, which class pairs actually dominate
the error budget, what the Option A / Option B gap actually came out to. The
physics statements are a lookup keyed on those observed facts, so the prose can
never drift away from the tables above it.
"""

import numpy as np
import pandas as pd


# What each feature means physically, and why a class would score high or low on it.
FEATURE_PHYSICS = {
    'rise_time': (
        "time from first detection to peak. Set by the photon diffusion time through the "
        "ejecta, so it scales with ejecta mass and inversely with expansion velocity: "
        "stripped-envelope SNe (Ib/Ic) rise in ~10-20 d, SLSNe take weeks-to-months because "
        "of their much larger ejecta masses and additional central-engine input, and stellar "
        "flares rise in minutes because the energy is released impulsively by magnetic "
        "reconnection rather than diffusing outward."),
    'decay_time': (
        "time from peak back down the light curve. For thermonuclear and core-collapse SNe "
        "this tracks radioactive decay of 56Ni -> 56Co -> 56Fe, giving the characteristic "
        "weeks-to-months tail; AGN never really 'decay' at all because their variability is "
        "stochastic accretion-disc flickering with no single peak; TDE fallback follows the "
        "canonical t^-5/3 decline, which is shallower than a SN tail."),
    'amplitude': (
        "peak-to-baseline change in relative flux. Separates explosive events (many "
        "magnitudes) from AGN stochastic variability (typically tenths of a magnitude "
        "about a persistent bright nucleus)."),
    'peak_val': (
        "peak relative flux. Because it is measured relative to each object's own baseline "
        "rather than as an absolute magnitude, it carries less information than it appears "
        "to; without a redshift-based distance correction it cannot express intrinsic "
        "luminosity, which is what actually distinguishes SLSNe from normal SNe."),
    'color_g_r': (
        "g-r colour at peak, i.e. photospheric temperature. Hot young SN photospheres and "
        "the very hot, near-constant-temperature TDE continuum are blue; AGN are redder and "
        "their colour barely changes; SNe redden steadily as the ejecta cool and expand."),
    'has_color': (
        "whether a two-band ZTF g-r colour was measurable at all. This is an *observational* "
        "flag, not an astrophysical property: TESS flares are single-band by construction, "
        "so it is always 0 for that class and 1 for most ZTF objects."),
}

# Physics of the confusions we expect to see, keyed on an unordered class pair.
CONFUSION_PHYSICS = {
    frozenset({'SNe', 'TDE'}): (
        "TDEs and SNe are the hardest coarse pair, and genuinely so: both are single, "
        "smooth, luminous flares on a galaxy nucleus with comparable rise times and "
        "amplitudes. What actually separates them is a near-constant blue colour and a "
        "t^-5/3 decline for TDEs versus steady reddening and a radioactive tail for SNe — "
        "both of which need well-sampled two-band photometry and a longer baseline than "
        "these summary shape features encode."),
    frozenset({'SNe', 'AGN'}): (
        "AGN contamination of the SN class usually means a poorly-sampled light curve where "
        "a stochastic AGN excursion was caught near a local maximum and looks like a single "
        "peak. More epochs, or an explicit variability/periodicity statistic, is the fix."),
    frozenset({'AGN', 'TDE'}): (
        "Both live in galactic nuclei and both can show slow, long-timescale variability; "
        "TDE searches in practice suffer exactly this contamination."),
    frozenset({'SN_Ib', 'SN_Ic'}): (
        "Ib versus Ic is defined *spectroscopically*, by the presence or absence of helium "
        "lines. Their light curves come from nearly the same explosion physics — similar "
        "ejecta masses, similar 56Ni yields, similar timescales — so photometric shape "
        "features contain little of the information the label is actually based on. This is "
        "a ceiling imposed by the labelling scheme, not a modelling failure."),
    frozenset({'SN_Ia', 'SN_Ic'}): (
        "Both are compact, fast-evolving explosions with similar rise times; separating them "
        "photometrically leans on the secondary near-infrared maximum and on colour "
        "evolution, neither of which survives reduction to four shape scalars."),
    frozenset({'SN_II', 'SN_Ia'}): (
        "SNe II retain a hydrogen envelope and many show a plateau, which should make them "
        "the most separable subtype from Ia on decay shape; residual confusion usually comes "
        "from IIb/IIn sub-subtypes folded into the same bucket."),
    frozenset({'SLSN', 'SN_II'}): (
        "SLSNe are separable mainly by being far more luminous and far slower; on "
        "baseline-relative flux, without an absolute-magnitude correction, that luminosity "
        "advantage is partly thrown away and only the timescale survives."),
}


def _fmt_pct(x):
    return 'n/a' if x is None or (isinstance(x, float) and np.isnan(x)) else f"{100 * x:.1f}%"


def _md_table(df, floatfmt='{:.3f}'):
    d = df.copy()
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].map(lambda v: '' if pd.isna(v) else floatfmt.format(v))
    header = '| ' + ' | '.join(str(c) for c in d.columns) + ' |'
    sep = '| ' + ' | '.join('---' for _ in d.columns) + ' |'
    body = '\n'.join('| ' + ' | '.join(str(v) for v in row) + ' |'
                     for row in d.itertuples(index=False))
    return '\n'.join([header, sep, body])


def build_results_markdown(count_report, split, stage1, stage2a, stage2b, comparison,
                           perm_s1, perm_s2, misclass_s1, misclass_s2,
                           recall_s1, recall_s2, resolution_s1, resolution_s2,
                           repeated_cv_s2=None, feature_df=None):
    """Assemble the whole results summary as a markdown string."""
    L = []
    add = L.append

    # ---------------------------------------------------------------- counts
    add("# Results summary\n")
    add("## Phase 2 — final sample\n")
    shortfalls = count_report[count_report['shortfall'] > 0]
    add(_md_table(count_report, '{:.0f}') + '\n')
    if len(shortfalls) == 0:
        add("All targets met: 150 objects in each of the four coarse classes (600 total), "
            "and 30 in each of the five SN subtypes. Every object counted here survived "
            "cross-match, download, cleaning, GP interpolation *and* feature extraction — "
            "the retry loop kept pulling candidates until that was true, so these are exact "
            "counts rather than whatever happened to survive.\n")
    else:
        add("**Shortfalls remain:**\n")
        for _, r in shortfalls.iterrows():
            add(f"- `{r['class']}`: {int(r['count'])}/{int(r['target'])} "
                f"(short by {int(r['shortfall'])})")
        add("\nA shortfall here means the candidate pool was genuinely exhausted, not that "
            "objects were silently dropped: raise the pool size / star list and re-run, and "
            "the loop resumes from its checkpoint.\n")

    n_train, n_test = len(split['idx_train']), len(split['idx_test'])
    add(f"\n**Split.** One stratified 80/20 split over all {n_train + n_test} rows "
        f"({n_train} train / {n_test} test), stratified on the coarse label. Stage 2's rows "
        f"are the SN-labelled subsets of those same two partitions, so a Stage 1 test object "
        f"cannot appear in Stage 2 training. This is asserted in code, not assumed.\n")

    # ---------------------------------------------------------------- stage 1
    add("\n## Phase 3, Stage 1 — coarse classification\n")
    add(_md_table(stage1['results']) + '\n')
    s1r = stage1['results'].set_index('Model')['Test accuracy']
    best1, worst1 = s1r.idxmax(), s1r.idxmin()
    add(f"\nBest: **{best1}** at {_fmt_pct(s1r.max())}; weakest: {worst1} at "
        f"{_fmt_pct(s1r.min())}. All four sit well above the {_fmt_pct(0.25)} chance rate for "
        f"four balanced classes.\n")
    add(f"\nMeasurement resolution: the Stage 1 test set holds {resolution_s1['n_test']} objects, "
        f"so accuracy moves in steps of {_fmt_pct(resolution_s1['accuracy_step'])} and carries a "
        f"worst-case standard error of about {_fmt_pct(resolution_s1['worst_case_std_error'])}. "
        f"Differences between the four models smaller than that are not real differences.\n")
    add("\n**Per-class recall (Random Forest):**\n")
    add(_md_table(recall_s1.reset_index()) + '\n')
    if len(misclass_s1):
        add("\n**Dominant confusions:**\n")
        add(_md_table(misclass_s1) + '\n')
        top = misclass_s1.iloc[0]
        phys = CONFUSION_PHYSICS.get(frozenset({top['true'], top['pred']}))
        add(f"\nThe single largest error mode is {top['true']} -> {top['pred']} "
            f"({int(top['n'])} objects, {_fmt_pct(top['share_of_errors'])} of all Stage 1 errors).")
        if phys:
            add(f" {phys}\n")

    # ---------------------------------------------------------------- stage 2
    add("\n## Phase 3, Stage 2 — SN subtype classification\n")
    add("### Option A — ground-truth routing (the ceiling)\n")
    add(_md_table(stage2a['results']) + '\n')
    add("\n### Option B — Stage-1-predicted SNe (the realistic path)\n")
    add(_md_table(stage2b['summary']) + '\n')
    add("\n### Option A vs Option B\n")
    add(_md_table(comparison) + '\n')

    a = comparison.set_index('Model')['Option A (ground-truth routing)']
    gap = comparison.set_index('Model')['Routing cost (A - B comparable)']
    add(f"\nThe honest comparison is the middle column: Option A and 'Option B comparable' "
        f"score the *same* population (true SN test objects) the same way, the only "
        f"difference being that Option B has to survive Stage 1's routing first. "
        f"That routing costs between {_fmt_pct(gap.min())} and {_fmt_pct(gap.max())} of "
        f"accuracy depending on the model — this gap is the price of the hierarchy, and it is "
        f"why the end-to-end column must never be compared directly against Option A: "
        f"end-to-end is computed over all {n_test} test objects including the easy non-SN "
        f"classes, which inflates it.\n")
    add(f"\nOption A tops out at {_fmt_pct(a.max())} ({a.idxmax()}). Subtyping is decisively "
        f"harder than the coarse problem ({_fmt_pct(s1r.max())}), and the reason is physical "
        f"rather than statistical — see the misclassification analysis below.\n")

    add(f"\n**Caveat — and this one is load-bearing.** The Stage 2 held-out set is only "
        f"{resolution_s2['n_test']} objects, about "
        f"{resolution_s2['mean_test_objects_per_class']:.0f} per subtype. A single accuracy on "
        f"that many draws has a worst-case standard error of roughly "
        f"{_fmt_pct(resolution_s2['worst_case_std_error'])}, and a per-subtype recall moves in "
        f"steps of {_fmt_pct(resolution_s2['per_class_recall_step'])}. A subtype recall of 0.00 "
        f"or 1.00 at this sample size is not evidence of anything. Balancing the dataset fixed "
        f"the *bias* from uneven attrition; it could not fix the variance, because 30 objects "
        f"per subtype is simply a small sample.\n")
    if repeated_cv_s2 is not None:
        add("\nRepeated stratified CV over the Stage 2 training partition — many more fits, "
            "no contact with Stage 1's test partition — gives a better-resolved estimate:\n")
        add(_md_table(repeated_cv_s2) + '\n')
        add("\nUse the held-out number as the untouched evaluation and this one to judge "
            "whether two models actually differ.\n")

    add("\n**Per-class recall, Option A (Random Forest):**\n")
    add(_md_table(recall_s2.reset_index()) + '\n')
    if len(misclass_s2):
        add("\n**Dominant subtype confusions:**\n")
        add(_md_table(misclass_s2) + '\n')
        for _, r in misclass_s2.head(3).iterrows():
            phys = CONFUSION_PHYSICS.get(frozenset({r['true'], r['pred']}))
            if phys:
                add(f"\n- **{r['true']} -> {r['pred']}** ({int(r['n'])} objects): {phys}")
        add("\n")

    # ---------------------------------------------------------------- phase 4
    add("\n## Phase 4 — feature importance and physical interpretation\n")
    for tag, perm in [('Stage 1', perm_s1), ('Stage 2', perm_s2)]:
        ranked = (perm.groupby('feature')['perm_importance'].mean()
                  .sort_values(ascending=False))
        add(f"\n### {tag} — most discriminative features\n")
        add(_md_table(ranked.reset_index().rename(
            columns={'perm_importance': 'mean permutation importance'})) + '\n')
        add(f"\nPermutation importance is computed on the held-out test set, identically for "
            f"all four models — including the bagged RBF SVM, which has no native importance "
            f"of any kind. That is the whole reason for using it: Gini importance and "
            f"logistic-regression coefficients are not comparable to each other, let alone to "
            f"a kernel machine.\n")
        add(f"\nThe top feature is **{ranked.index[0]}** — {FEATURE_PHYSICS.get(ranked.index[0], '')}\n")
        second = ranked.index[1]
        add(f"\nFollowed by **{second}** — {FEATURE_PHYSICS.get(second, '')}\n")
        near_zero = ranked[ranked <= 0.005]
        if len(near_zero):
            add(f"\nContributing essentially nothing at this stage: "
                f"{', '.join(f'`{f}`' for f in near_zero.index)}. A permutation importance at "
                f"or below zero means shuffling the column did not hurt accuracy — the model "
                f"was not using it.\n")

    if 'has_color' in perm_s1.groupby('feature')['perm_importance'].mean().nlargest(2).index:
        add("\n**A caveat about `has_color`.** It ranks near the top of Stage 1, but it is an "
            "observational artefact rather than astrophysics: it is 0 for every TESS flare and "
            "1 for essentially every ZTF object, so it partly encodes 'which survey did this "
            "come from'. The stellar-flare class is therefore easier than it looks. The "
            "physically meaningful Stage 1 result is the separation *among the three ZTF "
            "classes* (SNe / AGN / TDE), where `has_color` carries no information.\n")

    add("\n## Honest caveats\n")
    add("- **Feature set is deliberately minimal.** Four shape scalars plus a colour and a "
        "flag. The brief asks for peak magnitude, rise/decay times, amplitude and colour, and "
        "that is what is here — but subtype separation is known to need spectroscopic or "
        "richer photometric information (secondary maxima, plateau detection, absolute "
        "magnitude via redshift), so Stage 2's ceiling is set by the features, not the models.\n")
    add("- **Flux is baseline-relative, not absolute.** `peak_val` and `amplitude` are measured "
        "against each object's own baseline, so intrinsic luminosity — the thing that most "
        "cleanly separates SLSNe from normal SNe — is largely divided out. TNS redshifts are "
        "already downloaded and would enable an absolute-magnitude feature; that is the single "
        "highest-value addition available.\n")
    add("- **Flares come from a different instrument.** TESS flares are single-band, "
        "high-cadence, and drawn from a curated list of known active stars, whereas the other "
        "three classes are ZTF discoveries from TNS. Some of Stage 1's flare performance is "
        "survey signature rather than astrophysics.\n")
    add("- **Sub-subtypes are folded into parent buckets** (Ia-91bg, Iax -> SN_Ia; Ic-BL, Icn "
        "-> SN_Ic; IIb, IIn, IIP, IIL -> SN_II). This raises within-class variance, especially "
        "for SN_II, and some residual confusion is a direct consequence of that choice.\n")
    add("- **Small test sets**, as quantified above. Every accuracy in this notebook should be "
        "read with its error bar attached.\n")
    return '\n'.join(L)

In [ ]:
from IPython.display import Markdown

summary_md = build_results_markdown(
    count_report, split, stage1, stage2a, stage2b, comparison,
    perm_s1, perm_s2, misclass_s1, misclass_s2, recall_s1, recall_s2,
    res_s1, res_s2, repeated_cv_s2=repeated_cv_s2, feature_df=feature_df)

with open(os.path.join(FEATURE_DIR, 'results_summary.md'), 'w') as f:
    f.write(summary_md)
print('Written to', os.path.join(FEATURE_DIR, 'results_summary.md'))

Markdown(summary_md)